In [2]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from QuantNado.make_zarr_store import combine_cached_zarrs, process_bam

# Make Zarr Dataset
This notebook makes, loads, and explores the Zarr xarray dataset created from BAM files.

In [3]:
bam_files = sorted(
    glob.glob("data/2025-12-17_menin_inh_24hr/seqnado_output/**/aligned/SEM*.bam")
)
chrom_sizes_path = "data/hg38/hg38.chrom.sizes"
cache_dir = Path("example/zarr_cache")
cache_dir.mkdir(exist_ok=True, parents=True)

print(f"Found {len(bam_files)} BAM files")
print(f"Chromosome sizes: {chrom_sizes_path}")
print(f"Cache directory: {cache_dir}")

Found 15 BAM files
Chromosome sizes: data/hg38/hg38.chrom.sizes
Cache directory: example/zarr_cache


## Process BAM files to Zarr

Process each BAM file using the parallel chromosome processing:

In [ ]:
# Process each BAM file using BamZarrStore context manager
# Chromosomes are processed in parallel and incrementally appended to the zarr store
for bam_file in bam_files:
    sample_name = Path(bam_file).stem
    print(f"\nProcessing {sample_name}...")

    zarr_file = process_bam(
        bam_file=bam_file,
        chromsizes=chrom_sizes_path,
        cache_dir=cache_dir,
        max_workers=8,
    )

    print(f"  Saved to: {zarr_file}")

2025-12-18 16:54:24.949 | INFO     | QuantNado.make_zarr_store:process_bam:208 - Loaded 25 chromosomes from data/hg38/hg38.chrom.sizes
2025-12-18 16:54:24.950 | INFO     | QuantNado.make_zarr_store:process_bam:214 - Processing BAM file: data/2025-12-17_menin_inh_24hr/seqnado_output/atac/aligned/SEM-DMSO.bam for 25 chromosomes in parallel (max_workers=8)



Processing SEM-DMSO...


2025-12-18 16:54:27.150 | INFO     | QuantNado.make_zarr_store:__enter__:45 - Removed existing zarr file: example/zarr_cache/SEM-DMSO.zarr
2025-12-18 16:54:27.153 | INFO     | QuantNado.make_zarr_store:__enter__:50 - Created zarr store: example/zarr_cache/SEM-DMSO.zarr
2025-12-18 16:54:38.495 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr1: 77.63% sparse (max: 21404, dtype: uint16)
2025-12-18 16:54:38.910 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr8: 74.44% sparse (max: 39, dtype: uint16)
2025-12-18 16:54:38.956 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr7: 78.59% sparse (max: 49, dtype: uint16)
2025-12-18 16:54:39.009 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr5: 76.13% sparse (max: 64, dtype: uint16)
2025-12-18 16:54:39.026 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr6: 75.88% sparse (max: 41, dtype: uint16)
2025-12-18 16:54:40.864 | INFO     | QuantNado.make_

  Saved to: example/zarr_cache/SEM-DMSO.zarr

Processing SEM-MENi...


2025-12-18 16:55:38.705 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr1: 80.58% sparse (max: 27256, dtype: uint16)
2025-12-18 16:55:40.384 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr8: 75.56% sparse (max: 46, dtype: uint16)
2025-12-18 16:55:40.398 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr7: 81.52% sparse (max: 46, dtype: uint16)
2025-12-18 16:55:40.424 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr4: 78.23% sparse (max: 42, dtype: uint16)
2025-12-18 16:55:40.463 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr3: 77.29% sparse (max: 1977, dtype: uint16)
2025-12-18 16:55:40.517 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr6: 78.49% sparse (max: 44, dtype: uint16)
2025-12-18 16:55:40.542 | INFO     | QuantNado.make_zarr_store:process_chromosome:158 -   chr2: 77.65% sparse (max: 44, dtype: uint16)
2025-12-18 16:55:40.575 | INFO     | QuantNado.mak

In [ ]:
metadata_path = Path("example/combined_metadata.csv")
metadata_df = pd.read_csv(metadata_path)
output_path = Path("example/dataset.zarr")

# Combine all cached zarr files
combine_cached_zarrs(
    cache_dir=cache_dir,
    metadata_df=metadata_df,
    output_path=output_path
)

print(f"\nCombined dataset saved to {output_path}")

# Load Dataset

In [ ]:
output_path = Path("example/dataset.zarr")
ds = xr.open_zarr(output_path)

# Explore Zarr Dataset



In [ ]:
# Check dimensions and coordinates
print("Dimensions:", ds.dims)
print("\nCoordinates:", list(ds.coords))
print("\nData variables:", list(ds.data_vars))
print("\nAttributes:", ds.attrs)

## PCA analysis

Perform Principal Component Analysis on the entire dataset to visualize sample relationships:

In [ ]:
def prepare_pca_data(ds, subsample_positions=10000):
    """
    Prepare dataset for PCA by flattening and subsampling.
    
    Parameters:
    - ds: xarray Dataset with signal
    - subsample_positions: Number of random positions to sample (reduces memory)
    
    Returns:
    - PCA-transformed data, explained variance, and feature names
    """
    # Get signal data (samples x chromosomes x positions)
    signal_data = ds['signal']
    
    # Flatten chromosomes and positions into features
    # Result: samples x (chromosomes * positions)
    print("Flattening dataset...")
    n_samples = signal_data.sizes['sample']
    n_chroms = signal_data.sizes['chromosome']
    n_positions = signal_data.sizes['position']
    
    print(f"Original shape: {n_samples} samples x {n_chroms} chromosomes x {n_positions} positions")
    
    # Reshape to (samples, features)
    # This stacks all chromosomes and positions into one long feature vector
    signal_flat = signal_data.values.reshape(n_samples, -1)
    
    print(f"Flattened shape: {signal_flat.shape}")
    
    # Subsample positions to reduce memory usage
    if signal_flat.shape[1] > subsample_positions:
        print(f"Subsampling to {subsample_positions} random positions...")
        np.random.seed(42)
        subsample_idx = np.random.choice(signal_flat.shape[1], subsample_positions, replace=False)
        signal_flat = signal_flat[:, subsample_idx]
        print(f"Subsampled shape: {signal_flat.shape}")
    
    return signal_flat

def compute_pca(signal_flat, n_components=5):
    """
    Compute PCA on flattened signal data.
    
    Parameters:
    - signal_flat: Array of shape (n_samples, n_features)
    - n_components: Number of principal components to compute
    
    Returns:
    - pca_result: PCA-transformed data
    - pca: PCA object with explained variance
    """
    print(f"\nStandardizing data...")
    scaler = StandardScaler()
    signal_scaled = scaler.fit_transform(signal_flat)
    
    print(f"Computing PCA with {n_components} components...")
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(signal_scaled)
    
    print(f"\nExplained variance ratio:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"  PC{i+1}: {var*100:.2f}%")
    print(f"  Total: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    
    return pca_result, pca

# Prepare and compute PCA
print("Preparing data for PCA...")
signal_flat = prepare_pca_data(ds, subsample_positions=50000)

pca_result, pca = compute_pca(signal_flat, n_components=5)

print(f"\nPCA result shape: {pca_result.shape}")

### Plot PCA results

Visualize samples in PC space:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: PC1 vs PC2
ax = axes[0]
scatter = ax.scatter(
    pca_result[:, 0], 
    pca_result[:, 1],
    s=100,
    alpha=0.7,
    edgecolors='black',
    linewidth=1
)

# Add sample labels
for i, sample in enumerate(ds.sample.values):
    ax.annotate(
        sample, 
        (pca_result[i, 0], pca_result[i, 1]),
        fontsize=8,
        alpha=0.8,
        xytext=(5, 5),
        textcoords='offset points'
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA: PC1 vs PC2')
ax.grid(True, alpha=0.3)

# Plot 2: Scree plot (variance explained)
ax = axes[1]
pc_labels = [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))]
ax.bar(pc_labels, pca.explained_variance_ratio_ * 100, alpha=0.7, edgecolor='black')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Create PCA DataFrame for easier analysis
pca_df = pd.DataFrame(
    pca_result,
    columns=[f'PC{i+1}' for i in range(pca_result.shape[1])],
    index=ds.sample.values
)

print("\nPCA coordinates:")
print(pca_df)

### PCA with metadata coloring

If your dataset has metadata coordinates, you can color points by groups:

In [ ]:
# Check if we have metadata to use for coloring
metadata_coords = [c for c in ds.coords if c not in ['sample', 'chromosome', 'position']]

if metadata_coords:
    print(f"Available metadata for coloring: {metadata_coords}")
    
    # Example: Color by first metadata coordinate
    color_by = metadata_coords[0]
    colors = ds.coords[color_by].values
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Check if colors are categorical or numeric
    if np.issubdtype(colors.dtype, np.number):
        # Numeric coloring
        scatter = ax.scatter(
            pca_result[:, 0], 
            pca_result[:, 1],
            c=colors,
            s=100,
            alpha=0.7,
            cmap='viridis',
            edgecolors='black',
            linewidth=1
        )
        plt.colorbar(scatter, ax=ax, label=color_by)
    else:
        # Categorical coloring
        unique_vals = np.unique(colors)
        cmap = plt.cm.get_cmap('tab10', len(unique_vals))
        
        for i, val in enumerate(unique_vals):
            mask = colors == val
            ax.scatter(
                pca_result[mask, 0], 
                pca_result[mask, 1],
                c=[cmap(i)],
                label=val,
                s=100,
                alpha=0.7,
                edgecolors='black',
                linewidth=1
            )
        ax.legend(title=color_by, bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Add sample labels
    for i, sample in enumerate(ds.sample.values):
        ax.annotate(
            sample, 
            (pca_result[i, 0], pca_result[i, 1]),
            fontsize=8,
            alpha=0.8,
            xytext=(5, 5),
            textcoords='offset points'
        )
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'PCA colored by {color_by}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No metadata coordinates found for coloring")

# Reduce by ranges

In [ ]:
promoters_bed = "data/hg38/promoters_1024bp.bed"

# Load promoter regions from BED file
promoters_df = pd.read_csv(
    promoters_bed,
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "gene", "score", "strand"]
)

print(f"Loaded {len(promoters_df)} promoter regions")
print(f"Chromosomes: {promoters_df['chrom'].unique()}")
promoters_df.head()

## Extract signal for promoter regions

Now we'll extract the signal values for each promoter region and create a reduced dataset:

In [ ]:
def extract_promoter_signals(ds, promoters_df):
    """
    Extract signal for all promoter regions from the dataset.
    
    Returns a new dataset with dimensions: (sample, promoter, position)
    where position ranges from 0 to promoter_width for each promoter.
    """
    promoter_signals = []
    valid_promoters = []
    
    for idx, row in promoters_df.iterrows():
        chrom = row['chrom']
        start = row['start']
        end = row['end']
        gene = row['gene']
        
        # Check if chromosome exists in dataset
        if chrom not in ds.chromosome.values:
            continue
            
        try:
            # Extract signal for this region across all samples
            # Select chromosome and position range
            region_signal = ds.sel(
                chromosome=chrom,
                position=slice(start, end)
            )['signal']
            
            # Only keep if we got the expected size
            if region_signal.sizes['position'] == (end - start):
                promoter_signals.append(region_signal)
                valid_promoters.append({
                    'gene': gene,
                    'chrom': chrom,
                    'start': start,
                    'end': end,
                    'strand': row['strand']
                })
        except Exception as e:
            # Skip promoters that cause issues
            continue
    
    if not promoter_signals:
        raise ValueError("No valid promoters found in dataset")
    
    # Combine all promoter signals
    print(f"Combining {len(promoter_signals)} promoter regions...")
    combined = xr.concat(promoter_signals, dim='promoter')
    
    # Create a new position coordinate (relative to promoter start)
    promoter_width = promoter_signals[0].sizes['position']
    combined = combined.assign_coords(
        position=np.arange(promoter_width)
    )
    
    # Add promoter metadata as coordinates
    promoter_info_df = pd.DataFrame(valid_promoters)
    combined = combined.assign_coords(
        gene=('promoter', promoter_info_df['gene'].values),
        promoter_chrom=('promoter', promoter_info_df['chrom'].values),
        promoter_start=('promoter', promoter_info_df['start'].values),
        promoter_end=('promoter', promoter_info_df['end'].values),
        promoter_strand=('promoter', promoter_info_df['strand'].values)
    )
    
    return combined

# Extract promoter signals
print("Extracting promoter signals from dataset...")
promoter_ds = extract_promoter_signals(ds, promoters_df)

print(f"\nPromoter dataset shape: {promoter_ds.shape}")
print(f"Dimensions: {dict(promoter_ds.sizes)}")
print(f"Coordinates: {list(promoter_ds.coords)}")
promoter_ds

## Compute summary statistics

Calculate mean signal across promoters for each sample:

In [ ]:
# Calculate mean signal per promoter (averaged across position)
promoter_means = promoter_ds.mean(dim='position')

print(f"Promoter means shape: {promoter_means.shape}")
print(f"Dimensions: {dict(promoter_means.sizes)}")

# Convert to DataFrame for easier analysis
promoter_summary = promoter_means.to_dataframe().reset_index()
promoter_summary.head(10)

## Visualize promoter signals

Plot average signal across all promoters:

In [ ]:
# Plot average signal profile across all promoters
avg_signal_by_sample = promoter_ds.mean(dim='promoter')

fig, ax = plt.subplots(figsize=(12, 6))

# Plot each sample
for i, sample in enumerate(avg_signal_by_sample.sample.values):
    signal = avg_signal_by_sample.sel(sample=sample)
    ax.plot(signal.position, signal.values, label=sample, alpha=0.7)

ax.set_xlabel('Position relative to promoter start (bp)')
ax.set_ylabel('Average signal')
ax.set_title('Average ChIP-seq signal across all promoters')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Query specific promoters

Example: Find and plot signal for specific genes:

In [ ]:
# Example: Find promoters for genes of interest
gene_name = "MYC"  # Change this to your gene of interest

# Find matching genes (partial match with startswith)
matching_genes = [g for g in promoter_ds.gene.values if g.startswith(gene_name)]

if matching_genes:
    print(f"Found {len(matching_genes)} genes matching '{gene_name}':")
    print(matching_genes[:10])  # Show first 10
    
    # Select the first matching gene and plot
    gene_to_plot = matching_genes[0]
    gene_idx = np.where(promoter_ds.gene.values == gene_to_plot)[0][0]
    gene_signal = promoter_ds.isel(promoter=gene_idx)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for sample in gene_signal.sample.values:
        signal = gene_signal.sel(sample=sample)
        ax.plot(signal.position, signal.values, label=sample, alpha=0.7)
    
    ax.set_xlabel('Position relative to promoter start (bp)')
    ax.set_ylabel('Signal')
    ax.set_title(f'ChIP-seq signal at {gene_to_plot} promoter')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"No genes found matching '{gene_name}'")

## Save promoter dataset

Optionally save the reduced promoter dataset for faster loading:

In [ ]:
# Save promoter dataset
promoter_output = Path("example/promoter_dataset.zarr")

promoter_ds_to_save = promoter_ds.to_dataset(name='signal')

encoding = {
    'signal': {'chunks': (1, 100, 512)}  # (sample, promoter, position)
}

promoter_ds_to_save.to_zarr(
    promoter_output,
    mode='w',
    encoding=encoding,
    consolidated=False
)

print(f"Promoter dataset saved to {promoter_output}")
print(f"Original dataset size: {dict(ds.sizes)}")
print(f"Reduced dataset size: {dict(promoter_ds_to_save.sizes)}")

# Gene-level feature counts from GTF

Extract read counts for gene features from a GTF annotation file:

In [ ]:
def parse_gtf_attributes(attr_string):
    """Parse GTF attribute string into a dictionary."""
    attrs = {}
    for item in attr_string.strip().split(';'):
        item = item.strip()
        if item:
            key_value = item.split(' ', 1)
            if len(key_value) == 2:
                key, value = key_value
                attrs[key] = value.strip('"')
    return attrs

def load_gtf_genes(gtf_file, feature_type='gene'):
    """
    Load gene features from a GTF file.
    
    Parameters:
    - gtf_file: Path to GTF file
    - feature_type: Feature type to extract ('gene', 'exon', 'transcript', etc.)
    
    Returns:
    - DataFrame with gene information
    """
    genes = []
    
    with open(gtf_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            
            chrom, source, ftype, start, end, score, strand, frame, attributes = fields
            
            if ftype != feature_type:
                continue
            
            # Parse attributes
            attrs = parse_gtf_attributes(attributes)
            
            genes.append({
                'chrom': chrom,
                'start': int(start) - 1,  # GTF is 1-based, convert to 0-based
                'end': int(end),
                'strand': strand,
                'gene_id': attrs.get('gene_id', ''),
                'gene_name': attrs.get('gene_name', attrs.get('gene_id', '')),
                'gene_type': attrs.get('gene_type', attrs.get('gene_biotype', '')),
            })
    
    return pd.DataFrame(genes)

# GTF file path
gtf_file = "data/hg38/hg38.ncbiRefSeq.gtf"

print(f"Loading genes from GTF: {gtf_file}")
genes_df = load_gtf_genes(gtf_file, feature_type='gene')

print(f"\nLoaded {len(genes_df)} genes")
print(f"Chromosomes: {sorted(genes_df['chrom'].unique())}")
print(f"Gene types: {genes_df['gene_type'].value_counts()[:10]}")
print("\nFirst few genes:")
genes_df.head()

## Compute feature counts

Calculate total read counts for each gene across all samples:

In [ ]:
def compute_feature_counts(ds, genes_df, filter_gene_types=None):
    """
    Compute feature counts (total reads per gene) from the dataset.
    
    Parameters:
    - ds: xarray Dataset with signal
    - genes_df: DataFrame with gene annotations from GTF
    - filter_gene_types: List of gene types to include (e.g., ['protein_coding'])
                         If None, includes all genes
    
    Returns:
    - DataFrame with counts (genes x samples)
    """
    # Filter genes by type if requested
    if filter_gene_types:
        genes_df = genes_df[genes_df['gene_type'].isin(filter_gene_types)].copy()
        print(f"Filtering to {len(genes_df)} genes of types: {filter_gene_types}")
    
    counts_list = []
    valid_genes = []
    
    print(f"Computing counts for {len(genes_df)} genes...")
    
    for idx, gene in genes_df.iterrows():
        chrom = gene['chrom']
        start = gene['start']
        end = gene['end']
        gene_id = gene['gene_id']
        gene_name = gene['gene_name']
        
        # Skip if chromosome not in dataset
        if chrom not in ds.chromosome.values:
            continue
        
        try:
            # Extract signal for this gene region
            gene_signal = ds.sel(
                chromosome=chrom,
                position=slice(start, end)
            )['signal']
            
            # Sum signal across all positions to get total count per sample
            gene_counts = gene_signal.sum(dim='position')
            
            counts_list.append(gene_counts.values)
            valid_genes.append({
                'gene_id': gene_id,
                'gene_name': gene_name,
                'gene_type': gene['gene_type'],
                'chrom': chrom,
                'start': start,
                'end': end,
                'strand': gene['strand'],
                'length': end - start
            })
            
        except Exception as e:
            continue
        
        # Progress indicator
        if (len(valid_genes) % 1000) == 0:
            print(f"  Processed {len(valid_genes)} genes...")
    
    print(f"Successfully computed counts for {len(valid_genes)} genes")
    
    # Create counts DataFrame
    counts_array = np.array(counts_list)  # Shape: (n_genes, n_samples)
    counts_df = pd.DataFrame(
        counts_array,
        columns=ds.sample.values
    )
    
    # Add gene metadata
    gene_info_df = pd.DataFrame(valid_genes)
    counts_df = pd.concat([gene_info_df, counts_df], axis=1)
    
    return counts_df

# Compute counts for protein-coding genes
print("Computing feature counts...")
feature_counts = compute_feature_counts(
    ds, 
    genes_df,
    filter_gene_types=['protein_coding']  # Change to None for all gene types
)

print(f"\nFeature counts shape: {feature_counts.shape}")
print(f"Columns: {list(feature_counts.columns)}")
feature_counts.head()

## Normalize counts

Calculate RPKM (Reads Per Kilobase per Million mapped reads):

In [ ]:
def compute_rpkm(counts_df, sample_columns):
    """
    Compute RPKM (Reads Per Kilobase per Million mapped reads).
    
    RPKM = (counts / gene_length_kb) / (total_counts_millions)
    
    Parameters:
    - counts_df: DataFrame with counts and 'length' column
    - sample_columns: List of column names containing count data
    
    Returns:
    - DataFrame with RPKM values
    """
    rpkm_df = counts_df.copy()
    
    # Gene length in kilobases
    gene_length_kb = counts_df['length'] / 1000
    
    for sample in sample_columns:
        # Total counts per sample in millions
        total_counts_millions = counts_df[sample].sum() / 1e6
        
        # Calculate RPKM
        rpkm_df[sample] = (counts_df[sample] / gene_length_kb) / total_counts_millions
    
    return rpkm_df

# Get sample column names (exclude metadata columns)
metadata_cols = ['gene_id', 'gene_name', 'gene_type', 'chrom', 'start', 'end', 'strand', 'length']
sample_cols = [col for col in feature_counts.columns if col not in metadata_cols]

print(f"Computing RPKM for {len(sample_cols)} samples...")
rpkm_df = compute_rpkm(feature_counts, sample_cols)

print(f"\nRPKM shape: {rpkm_df.shape}")
print("\nRPKM values (first few genes):")
rpkm_df.head()

## Save feature counts

Save counts and RPKM to CSV files:

In [ ]:
# Save to CSV
counts_output = Path("example/feature_counts.csv")
rpkm_output = Path("example/feature_counts_rpkm.csv")

feature_counts.to_csv(counts_output, index=False)
rpkm_df.to_csv(rpkm_output, index=False)

print(f"Saved raw counts to: {counts_output}")
print(f"Saved RPKM values to: {rpkm_output}")

# Summary statistics
print(f"\nSummary:")
print(f"  Total genes: {len(feature_counts)}")
print(f"  Total samples: {len(sample_cols)}")
print(f"  Mean counts per gene: {feature_counts[sample_cols].mean(axis=1).mean():.2f}")
print(f"  Mean RPKM per gene: {rpkm_df[sample_cols].mean(axis=1).mean():.2f}")

## Visualize count distribution

Plot distribution of counts across genes and samples:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total counts per sample
sample_totals = feature_counts[sample_cols].sum(axis=0)
axes[0].bar(range(len(sample_totals)), sample_totals.values)
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Total counts')
axes[0].set_title('Total read counts per sample')
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribution of log2(RPKM + 1) for first sample
sample_to_plot = sample_cols[0]
rpkm_values = rpkm_df[sample_to_plot].values
log_rpkm = np.log2(rpkm_values + 1)

axes[1].hist(log_rpkm, bins=50, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('log2(RPKM + 1)')
axes[1].set_ylabel('Number of genes')
axes[1].set_title(f'RPKM distribution: {sample_to_plot}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show top expressed genes
print("\nTop 10 expressed genes (by mean RPKM):")
mean_rpkm = rpkm_df[sample_cols].mean(axis=1)
top_genes_idx = mean_rpkm.nlargest(10).index
print(rpkm_df.loc[top_genes_idx, ['gene_name', 'gene_type'] + sample_cols[:3]])